# Notebook 12 – Feature Leakage

## 1. What is Feature Leakage?

Feature leakage happens when information that should not be available to the Machine Learning model at prediction time is included in the training data.

Leakage can make the model look much better during training or testing than it actually is in the real world.

Common types include:
- Target Leakage
- Train-Test Leakage
- Temporal Leakage
- Post-Outcome Features
- Leakage Through Aggregations
- Leakage Through Target Encoding

In [1]:
import pandas as pd

df = pd.read_csv("Titanic-Dataset.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. Target Leakage

Target leakage occurs when a feature contains information that is directly or indirectly derived from the target variable.

For example, if we create a feature using `Survived` and then use that feature to predict `Survived`, the model is receiving the answer as an input.

This can produce unrealistically high model performance.

In [2]:
# Incorrect: feature directly uses the target

df["Survival_Info"] = df["Survived"]

df[["Survived", "Survival_Info"]].head()

,Survived,Survival_Info
0,0,0
1,1,1
2,1,1
3,1,1
4,0,0


## 3. Train-Test Leakage

Train-test leakage happens when information from the test set is used while preparing or training the model.

For example, calculating a statistic such as the mean or scaling parameters using the complete dataset before splitting can allow test-set information to influence training.

The correct approach is to:
1. Split the data first.
2. Fit preprocessing only on training data.
3. Apply the fitted transformation to the test data.

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[["Age", "Fare"]]
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train = X_train.fillna(X_train.median())
X_test = X_test.fillna(X_train.median())

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Temporal Leakage

Temporal leakage occurs when information from the future is used to predict an event that happened in the past.

For example, using a customer's future purchase information to predict whether the customer will make a purchase today would create leakage.

For time-dependent data, features should only use information that was available at the prediction time.

In [7]:
# Example of the correct idea

df["Event_Date"] = pd.to_datetime("1912-04-15")

df[["Event_Date"]].head()

,Event_Date
0,1912-04-15
1,1912-04-15
2,1912-04-15
3,1912-04-15
4,1912-04-15


## 5. Post-Outcome Features

A post-outcome feature is created using information that becomes available after the target event has already happened.

Such features should not be used for prediction because they reveal information from after the outcome.

Example:

If we are predicting whether a customer will cancel a service, using `Cancellation_Reason` would create leakage because the reason is known only after cancellation.

In [8]:
# Example concept

example = pd.DataFrame({
    "Customer": ["A", "B", "C"],
    "Cancelled": [1, 0, 1],
    "Cancellation_Reason": ["Price", None, "Service"]
})

example

,Customer,Cancelled,Cancellation_Reason
0,A,1,Price
1,B,0,NaN
2,C,1,Service


## 6. Leakage Through Aggregations

Aggregated features can cause leakage when they are calculated using information from the target or from data that should not be available at prediction time.

For example, calculating the average survival rate for a ticket using the complete dataset would allow information from other passengers' outcomes to influence the feature.

Aggregation features should be calculated using only appropriate training information.

In [9]:
# Incorrect: uses target information from the complete dataset

ticket_survival = df.groupby("Ticket")["Survived"].mean()

df["Ticket_Survival_Rate"] = df["Ticket"].map(ticket_survival)

df[["Ticket", "Survived", "Ticket_Survival_Rate"]].head()

,Ticket,Survived,Ticket_Survival_Rate
0,A/5 21171,0,0.0
1,PC 17599,1,1.0
2,STON/O2. 3101282,1,1.0
3,113803,1,0.5
4,373450,0,0.0


## 7. Leakage Through Target Encoding

Target Encoding replaces a category with a statistic calculated from the target.

For example, replacing `Sex` with the average `Survived` value for each sex.

If this encoding is calculated using the complete dataset before splitting, target information from the test data can leak into the training data.

Target encoding should be calculated using training data only and preferably with an out-of-fold approach during model training.

In [10]:
# Correct idea: calculate encoding from training data only

train_data = pd.DataFrame({
    "Sex": ["male", "female", "male", "female"],
    "Survived": [0, 1, 0, 1]
})

target_mean = train_data.groupby("Sex")["Survived"].mean()

target_mean

Sex
female    1.0
male      0.0
Name: Survived, dtype: float64

## 8. Incorrect Feature Engineering → Leakage

The following example creates a feature using the target variable.

This is incorrect because `Survived` is the target we are trying to predict.

### Problem
The feature contains the answer.

### Result
The model can learn the target directly, producing misleading performance.

### Decision
Remove the feature.

In [11]:
df["Leaked_Feature"] = df["Survived"]

df[["Survived", "Leaked_Feature"]].head()

,Survived,Leaked_Feature
0,0,0
1,1,1
2,1,1
3,1,1
4,0,0


## 9. Correct Feature Engineering → No Leakage

A feature is safe when it uses only information that would be available when the prediction is made.

For example, `FamilySize` uses `SibSp` and `Parch`. These columns do not contain the target information.

Therefore, this feature can be safely created before model training.

In [12]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

df[["SibSp", "Parch", "FamilySize"]].head()

,SibSp,Parch,FamilySize
0,1,0,2
1,1,0,2
2,0,0,1
3,1,0,2
4,0,0,1


## 10. Detecting Feature Leakage

Feature leakage can be detected by checking:

- Whether a feature directly uses the target.
- Whether a feature is created after the target outcome.
- Whether test data was used during preprocessing.
- Whether future information is used.
- Whether aggregations contain target information.
- Whether target encoding was calculated using the complete dataset.
- Whether model performance is unrealistically high.

The most important question is:

**Would this information actually be available at prediction time?**

In [13]:
target = "Survived"

leakage_features = [
    col for col in df.columns
    if target.lower() in col.lower()
]

print("Potential leakage features:", leakage_features)

Potential leakage features: ['Survived']


## 11. Preventing Feature Leakage

To prevent leakage:

1. Split the data before learning preprocessing parameters.
2. Fit transformations only on training data.
3. Do not use target-derived features as normal input features.
4. Avoid using future or post-outcome information.
5. Calculate aggregations using leakage-safe data.
6. Perform target encoding using training data only.
7. Use Pipelines to keep preprocessing and model training organized.

Preventing leakage gives a more realistic estimate of how the model will perform on unseen data.

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

X_train_ready = pipeline.fit_transform(X_train)
X_test_ready = pipeline.transform(X_test)

## 12. Required Documentation for Important Engineered Features

For every important engineered feature, document:

### Feature Name
Name of the newly created feature.

### Source Columns
Which columns were used?

### Logic
Explain the formula or business rule.

### Reason
Why was the feature created?

### Business Meaning
What does the feature represent?

### ML Relevance
How could the feature help a Machine Learning model?

### Leakage Check
Could this feature introduce data leakage?

### Final Decision
Choose one:
- Retain
- Remove
- Needs Further Analysis

In [16]:
documentation = pd.DataFrame([
    ["FamilySize", "SibSp, Parch", "SibSp + Parch + 1",
     "Represent family size", "Total family members",
     "May capture family-related patterns",
     "No target information used", "Retain"],

    ["Survival_Info", "Survived", "Copy of target",
     "Demonstrate target leakage", "Contains target information",
     "Creates unrealistic model performance",
     "Direct target leakage", "Remove"],

    ["Ticket_Survival_Rate", "Ticket, Survived",
     "Mean Survived by Ticket",
     "Demonstrate aggregation leakage",
     "Ticket-level survival information",
     "Could appear predictive",
     "Uses target information", "Remove"],

    ["FarePerPerson", "Fare, FamilySize",
     "Fare / FamilySize",
     "Represent fare per family member",
     "Approximate fare per person",
     "May provide useful fare information",
     "No target information used", "Needs Further Analysis"]
], columns=[
    "Feature Name",
    "Source Columns",
    "Logic",
    "Reason",
    "Business Meaning",
    "ML Relevance",
    "Leakage Check",
    "Final Decision"
])

documentation

,Feature Name,Source Columns,Logic,Reason,Business Meaning,ML Relevance,Leakage Check,Final Decision
0,FamilySize,"SibSp, Parch",SibSp + Parch + 1,Represent family size,Total family members,May capture family-related patterns,No target information used,Retain
1,Survival_Info,Survived,Copy of target,Demonstrate target leakage,Contains target information,Creates unrealistic model performance,Direct target leakage,Remove
2,Ticket_Survival_Rate,"Ticket, Survived",Mean Survived by Ticket,Demonstrate aggregation leakage,Ticket-level survival information,Could appear predictive,Uses target information,Remove
3,FarePerPerson,"Fare, FamilySize",Fare / FamilySize,Represent fare per family member,Approximate fare per person,May provide useful fare information,No target information used,Needs Further Analysis


## 13. Conclusion

Feature leakage happens when information that should not be available at prediction time enters the Machine Learning process.

In this notebook, we covered target leakage, train-test leakage, temporal leakage, post-outcome features, aggregation leakage, and target encoding leakage.

The key rule is:

**A feature should use only the information that would be available at the time of prediction.**

Correct feature engineering helps create reliable models, while leakage can make model performance look better than it really is.

In [17]:
print("Feature leakage analysis completed successfully.")

Feature leakage analysis completed successfully.
